# Mens rea by element: stacked-bar grid

Standalone program. Reads every offense CSV in `DATA_DIR` (columns `law_number`, `conduct_merged`, `result_merged`, `circumstance_merged`) and draws one stacked bar per element for each offense, ordered by the amount of stated mens rea data. A coded value of `n/a` is treated as **Silent**.

**Scales automatically.** Nothing in the code assumes a fixed number of offenses or jurisdictions. Add or remove CSVs and re-run: the counts, the grid (columns × rows), the panel text size and the file names all adjust. Optional settings in the first code cell let you force a column count, split the grid across several pages, or double-check the expected counts.

**To run:** put the CSVs in the same folder as this notebook (or set `DATA_DIR`), then *Run All*. Output goes to a `figures` folder next to this notebook, as PNG and SVG.

Requires `pandas`, `numpy`, `matplotlib`.

In [ ]:
import re
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

%matplotlib inline

# ---- input / output -----------------------------------------------------------------
DATA_DIR = Path(".")                               # folder containing the offense CSVs
FILE_GLOB = "*_rounded_down.csv"                   # which files count as offense files
NAME_PATTERN = r"-sol_(.+)_rounded_down\.csv$"     # offense name = the part captured in ( )
OUT_DIR = Path("figures")                          # output folder, created next to this notebook
OUT_DIR.mkdir(exist_ok=True)

# ---- optional sanity checks (None = accept whatever is in the folder) ----------------
EXPECTED_OFFENSES = None          # e.g. 50 -> stop with an error if a different number is read
EXPECTED_JURISDICTIONS = None     # e.g. 51

# ---- coding scheme --------------------------------------------------------------------
ELEMENTS = {                      # CSV column -> axis label, in plotting order
    "conduct_merged": "Conduct",
    "result_merged": "Result",
    "circumstance_merged": "Circumstance",
}
EL_LABELS = list(ELEMENTS.values())
EL_SHORT = {"Conduct": "Cond.", "Result": "Res.", "Circumstance": "Circ."}   # used if panels get narrow
LEVELS = ["Purposive", "Knowing", "Reckless", "Negligent", "Strict Liability", "Silent"]

# Palette sampled from the reference figure
COLORS = {
    "Purposive":        "#7A0030",
    "Knowing":          "#B8004A",
    "Reckless":         "#E6005D",
    "Negligent":        "#F27DA8",
    "Strict Liability": "#F9C2D5",
    "Silent":           "#D6D6D6",
}

# ---- layout ------------------------------------------------------------------------------
PAGE_SIZE = (8.5, 11)      # inches; portrait letter. Landscape: (11, 8.5)
PANELS_PER_PAGE = None     # None = every offense on one page; e.g. 24 = pages of 24
NCOLS = None               # None = choose columns automatically; or force a number, e.g. 4
FONT = 7                   # panel text size (pt) at full panel size (the 24-offense page)
MIN_FONT = 4.5             # panel text never shrinks below this as offenses are added
MAX_FONT = 9               # ...or grows above this when there are only a few offenses


In [ ]:
SMALL_WORDS = {"a", "an", "the", "to", "with", "of", "and", "or", "in", "on", "for"}

def display_name(stem: str) -> str:
    """'Driving_with_a_Suspended_License' -> 'Driving with a Suspended License',
    'Kidnapping_1st_degree' -> 'Kidnapping 1st Degree', 'DUI' stays 'DUI'."""
    words = stem.replace("_", " ").split()
    return " ".join(
        w.lower() if (i > 0 and w.lower() in SMALL_WORDS) else w[0].upper() + w[1:]
        for i, w in enumerate(words)
    )

# ---- read every matching file ----------------------------------------------------------
frames, skipped = [], []
for path in sorted(DATA_DIR.glob(FILE_GLOB)):
    m = re.search(NAME_PATTERN, path.name)
    if not m:
        skipped.append(path.name)
        continue
    # keep_default_na=False keeps the literal string "n/a" instead of turning it into NaN
    df = pd.read_csv(path, keep_default_na=False)
    missing_cols = {"law_number", *ELEMENTS} - set(df.columns)
    if missing_cols:
        raise ValueError(f"{path.name} is missing columns: {sorted(missing_cols)}")
    df["offense"] = display_name(m.group(1))
    df["source_file"] = path.name
    frames.append(df)

if skipped:
    print(f"WARNING: skipped {len(skipped)} file(s) whose names don't match NAME_PATTERN:\n  "
          + "\n  ".join(skipped))
if not frames:
    raise FileNotFoundError(f"No offense files matching {FILE_GLOB!r} in {DATA_DIR.resolve()}")

raw = pd.concat(frames, ignore_index=True)

# ---- consistency checks (all derived from the data, nothing hard-coded) ---------------
clash = raw.groupby("offense")["source_file"].unique().loc[lambda s: s.map(len) > 1]
if not clash.empty:
    raise ValueError("Different files produce the same offense name:\n"
                     + "\n".join(f"  {o}: {list(f)}" for o, f in clash.items()))

dupe_rows = raw.groupby("offense")["law_number"].agg(lambda s: int(s.duplicated().sum())).loc[lambda s: s > 0]
if not dupe_rows.empty:
    raise ValueError(f"Repeated law_number rows within an offense:\n{dupe_rows.to_string()}")

ALL_JURIS = sorted(raw["law_number"].unique())
N_JURIS = len(ALL_JURIS)
coverage = raw.groupby("offense")["law_number"].nunique()
incomplete = coverage[coverage != N_JURIS]
if not incomplete.empty:
    raise ValueError(f"These offenses don't cover all {N_JURIS} jurisdictions "
                     f"(rows found shown):\n{incomplete.to_string()}")

N_OFFENSES = raw["offense"].nunique()
if EXPECTED_OFFENSES is not None and N_OFFENSES != EXPECTED_OFFENSES:
    raise ValueError(f"Expected {EXPECTED_OFFENSES} offenses, found {N_OFFENSES}")
if EXPECTED_JURISDICTIONS is not None and N_JURIS != EXPECTED_JURISDICTIONS:
    raise ValueError(f"Expected {EXPECTED_JURISDICTIONS} jurisdictions, found {N_JURIS}")

long = (
    raw.melt(id_vars=["offense", "law_number"], value_vars=list(ELEMENTS),
             var_name="element", value_name="mens_rea")
       .assign(element=lambda d: d["element"].map(ELEMENTS),
               mens_rea=lambda d: d["mens_rea"].astype(str).str.strip().replace({"n/a": "Silent"}))
)
long["stated"] = long["mens_rea"] != "Silent"

unknown = set(long["mens_rea"]) - set(LEVELS)
if unknown:
    raise ValueError(f"Unexpected mens rea labels: {sorted(unknown)}")

print(f"OK: {N_OFFENSES} offenses x {N_JURIS} jurisdictions x {len(ELEMENTS)} elements "
      f"= {len(long):,} coded element-cells")

all_silent = long.groupby("offense")["stated"].sum().loc[lambda s: s == 0].index.tolist()
if all_silent:
    print(f"WARNING: no stated mens rea anywhere in: {', '.join(all_silent)} "
          "(check whether this is real silence or an extraction problem)")


In [ ]:
# Share (%) of jurisdictions at each level, per offense x element
counts = (
    long.groupby(["offense", "element", "mens_rea"]).size()
        .unstack("mens_rea", fill_value=0)
        .reindex(columns=LEVELS, fill_value=0)
)
shares = counts / N_JURIS * 100

# Order offenses by amount of stated mens rea data (non-silent cells, all elements)
stated_by_offense = long.groupby("offense")["stated"].sum()
order = stated_by_offense.sort_values(ascending=False, kind="stable").index.tolist()


In [ ]:
# ---- automatic layout -------------------------------------------------------------------
# Page margins (inches). Title and legend keep a fixed size; panel text scales with panel width.
TOP_IN, BOTTOM_IN, RIGHT_IN, GAP_W = 0.85, 0.8, 0.15, 0.12
REF_PANEL_W, REF_PANEL_H = 1.86, 1.108    # panel size on the original 24-offense page (FONT looks right here)
PANEL_ASPECT = REF_PANEL_W / REF_PANEL_H  # preferred width/height of a panel
CHAR_W_PER_PT = 0.0954 / 12               # approx. inches per character per point (for "Circumstance")

def _geometry(n, ncols, font, page_size):
    fig_w, fig_h = page_size
    nrows = int(np.ceil(n / ncols))
    left_in = 0.55 + 0.40 * (font - 7) / 7   # room for y tick labels and "Share (%)"
    gap_h = 0.54 + 0.54 * (font - 7) / 7    # room for x tick labels and up to 2-line panel titles
    ax_w = (fig_w - left_in - RIGHT_IN - (ncols - 1) * GAP_W) / ncols
    top_in = TOP_IN + 0.06 * max(0.0, font - 7)   # bigger panel titles need more headroom
    ax_h = (fig_h - top_in - BOTTOM_IN - (nrows - 1) * gap_h) / nrows
    return dict(n=n, nrows=nrows, ncols=ncols, font=font, left_in=left_in, gap_h=gap_h,
                top_in=top_in, ax_w=ax_w, ax_h=ax_h, page_size=page_size)

def _fit(n, ncols, page_size):
    # Panel font follows panel width; iterate because margins depend on the font
    font = FONT
    for _ in range(5):
        g = _geometry(n, ncols, font, page_size)
        font = round(float(np.clip(FONT * g["ax_w"] / REF_PANEL_W, MIN_FONT, MAX_FONT)), 1)
    return _geometry(n, ncols, font, page_size)

def choose_layout(n, page_size=PAGE_SIZE, ncols=NCOLS):
    """Pick the column count that gives the largest panels at a sensible shape."""
    if ncols:
        return _fit(n, ncols, page_size)
    candidates = [_fit(n, c, page_size) for c in range(1, n + 1)]
    candidates = [g for g in candidates if g["ax_h"] > 0 and g["ax_w"] > 0]
    if not candidates:
        raise ValueError(f"{n} panels can't fit on a {page_size} page; set PANELS_PER_PAGE")
    return max(candidates, key=lambda g: min(g["ax_w"], g["ax_h"] * PANEL_ASPECT))

def plot_page(offenses, g, subtitle_suffix=""):
    fig_w, fig_h = g["page_size"]
    nrows, ncols, font = g["nrows"], g["ncols"], g["font"]
    fig, axes = plt.subplots(nrows, ncols, figsize=g["page_size"], sharey=True, squeeze=False)
    fig.subplots_adjust(left=g["left_in"] / fig_w, right=1 - RIGHT_IN / fig_w,
                        top=1 - g["top_in"] / fig_h, bottom=BOTTOM_IN / fig_h,
                        wspace=GAP_W / g["ax_w"], hspace=g["gap_h"] / g["ax_h"])

    title_size = font * 1.25
    wrap_chars = max(12, int(g["ax_w"] * 72 / (title_size * 0.55)))
    tick_size = font * 0.85
    longest = max(EL_LABELS, key=len)
    use_short = len(longest) * CHAR_W_PER_PT * tick_size > 0.98 * g["ax_w"] / len(EL_LABELS)
    xlabels = [EL_SHORT.get(l, l) for l in EL_LABELS] if use_short else EL_LABELS

    x = np.arange(len(EL_LABELS))
    for ax, off in zip(axes.flat, offenses):
        tbl = shares.loc[off].reindex(EL_LABELS)
        bottom = np.zeros(len(EL_LABELS))
        for lvl in LEVELS:
            vals = tbl[lvl].to_numpy()
            ax.bar(x, vals, bottom=bottom, width=0.6, color=COLORS[lvl], linewidth=0)
            bottom += vals
        ax.set_title(textwrap.fill(off, wrap_chars), fontsize=title_size,
                     fontweight="bold", pad=3, linespacing=1.0)
        ax.set_xticks(x)
        ax.set_xticklabels(xlabels, fontsize=tick_size)
        ax.set_ylim(0, 100)
        ax.set_yticks(range(0, 101, 20))
        ax.tick_params(axis="y", labelsize=font)
        ax.tick_params(length=2.5 * font / FONT, width=0.6, pad=2)
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["left", "bottom"]].set_linewidth(0.6)

    for ax in axes[:, 0]:
        ax.set_ylabel("Share (%)", fontsize=font)
    for ax in axes.flat[len(offenses):]:
        ax.set_visible(False)

    # Page-level text keeps a fixed size however many panels there are
    fig.suptitle(
        f"Mens Rea by Element, Within Each Offense (% of {N_JURIS} jurisdictions)\n"
        f"offenses ordered by amount of stated mens rea data{subtitle_suffix}",
        fontsize=FONT * 1.6, fontweight="bold", y=1 - 0.3 / fig_h, va="top",
    )
    handles = [Patch(facecolor=COLORS[l], label=l) for l in LEVELS]
    fig.legend(handles=handles, loc="lower center", ncol=len(LEVELS), frameon=False,
               fontsize=FONT, handlelength=2.2, columnspacing=1.8,
               bbox_to_anchor=(0.5, 0.2 / fig_h))
    return fig


In [ ]:
per_page = PANELS_PER_PAGE or len(order)
pages = [order[i:i + per_page] for i in range(0, len(order), per_page)]
layout = choose_layout(len(pages[0]))          # same grid on every page

print(f"{len(order)} offenses -> {len(pages)} page(s), {layout['ncols']} columns x {layout['nrows']} rows, "
      f"panel text {layout['font']:.1f} pt")

for p, offs in enumerate(pages, start=1):
    multi = len(pages) > 1
    fig = plot_page(offs, layout, f" (page {p} of {len(pages)})" if multi else "")
    stem = f"fig_offense_by_element_all{len(order)}" + (f"_p{p}" if multi else "")
    # Saved at exact page size (no tight cropping) so it drops straight into a document
    fig.savefig(OUT_DIR / f"{stem}.png", dpi=300, facecolor="white")
    fig.savefig(OUT_DIR / f"{stem}.svg", facecolor="white")   # vector: stays sharp at any size
    print(f"Saved {(OUT_DIR / stem).resolve()}.png and .svg")
    plt.show()
